In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [15]:
aq = pd.read_csv("rawds/city_day.csv")          # or city_hour.csv if you want hourly
print("=== Air Quality Columns ===")
print(aq.columns.tolist())
print(aq.head(3))
aq_kolkata = aq[aq['City'] == 'Kolkata'].copy()
print("\nKolkata rows:", len(aq_kolkata))

=== Air Quality Columns ===
['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']
      City    Datetime  PM2.5   PM10     NO   NO2    NOx   NH3    CO   SO2  \
0    Delhi  2015-01-01  153.3  241.7  182.9  33.0   81.3  38.5  1.87  64.5   
1   Mumbai  2015-01-01   70.5  312.7  195.0  42.0  122.5  31.5  7.22  83.8   
2  Chennai  2015-01-01  174.1  275.4   56.2  68.8  230.9  28.5  8.56  60.8   

      O3  Benzene  Toluene  Xylene    AQI    AQI_Bucket  
0   83.6    18.93    20.81    8.32  204.5        Severe  
1  108.0     2.01    19.41    2.86   60.9  Satisfactory  
2   43.9    19.07    10.19    9.63  486.5        Severe  

Kolkata rows: 3653


In [17]:
weather = pd.read_csv("rawds/2005-2025_kolkata_hourly_weather_full_dataset.csv")
print("\n=== Weather Columns ===")
print(weather.columns.tolist())


=== Weather Columns ===
['date', 'temperature_2m', 'relative_humidity_2m', 'apparent_temperature', 'rain', 'pressure_msl', 'wind_speed_10m', 'dew_point_2m', 'weather_code', 'surface_pressure', 'soil_temperature_0_to_7cm', 'soil_moisture_0_to_7cm', 'wind_gusts_10m', 'wind_direction_10m', 'cloud_cover', 'precipitation', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit', 'cloud_cover_high', 'cloud_cover_low', 'cloud_cover_mid', 'soil_temperature_7_to_28cm', 'wind_speed_100m', 'wind_direction_100m', 'is_day', 'total_column_integrated_water_vapour', 'wet_bulb_temperature_2m', 'direct_radiation', 'direct_radiation_instant']


In [19]:
traffic = pd.read_csv("rawds/traffic.csv")
print("\n=== Traffic Columns ===")
print(traffic.columns.tolist())


=== Traffic Columns ===
['DateTime', 'Junction', 'Vehicles', 'ID']


In [27]:
aq_kolkata = aq_kolkata.rename(columns={'Datetime': 'datetime'})
aq_kolkata['datetime'] = pd.to_datetime(aq_kolkata['datetime']).dt.tz_localize(None)

In [26]:
weather = weather.rename(columns={'date': 'datetime'})
weather['datetime'] = pd.to_datetime(weather['datetime']).dt.tz_localize(None)

In [25]:
traffic = traffic.rename(columns={'DateTime': 'datetime'})
traffic['datetime'] = pd.to_datetime(traffic['datetime']).dt.tz_localize(None)

In [28]:
merged = pd.merge(aq_kolkata, weather, on='datetime', how='inner')

In [29]:
merged['hour'] = merged['datetime'].dt.hour
merged['weekday'] = merged['datetime'].dt.weekday
merged['is_weekend'] = merged['weekday'].isin([5,6]).astype(int)

merged['traffic_level'] = np.where(
    merged['hour'].isin([7,8,9,10,17,18,19,20,21]), 'High', 'Medium'
)

In [30]:
print("\nMerged Shape:", merged.shape)
print("Columns:", merged.columns.tolist())


Merged Shape: (3653, 48)
Columns: ['City', 'datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'temperature_2m', 'relative_humidity_2m', 'apparent_temperature', 'rain', 'pressure_msl', 'wind_speed_10m', 'dew_point_2m', 'weather_code', 'surface_pressure', 'soil_temperature_0_to_7cm', 'soil_moisture_0_to_7cm', 'wind_gusts_10m', 'wind_direction_10m', 'cloud_cover', 'precipitation', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit', 'cloud_cover_high', 'cloud_cover_low', 'cloud_cover_mid', 'soil_temperature_7_to_28cm', 'wind_speed_100m', 'wind_direction_100m', 'is_day', 'total_column_integrated_water_vapour', 'wet_bulb_temperature_2m', 'direct_radiation', 'direct_radiation_instant', 'hour', 'weekday', 'is_weekend', 'traffic_level']


In [31]:

merged.to_csv("kolkata_aqi_merged_final.csv", index=False)


print("\nAQI Summary:")
print(merged['AQI'].describe())


AQI Summary:
count    3653.000000
mean      250.635396
std       144.285499
min         0.100000
25%       125.400000
50%       249.500000
75%       376.700000
max       499.900000
Name: AQI, dtype: float64
